Flash Attention的两个贡献
- Tiled Attention其实就是Block Matrix–Matrix Multiplication
- Online/streaming softmax 是经典数值计算技巧，在统计建模和机器学习中早已出现
    - $\text{softmax}(x) = \text{softmax}(x - \max(x))$（（防止 $e^x$ 溢出））

### tiled(tilling) & softmax

$$
O = \text{softmax}(QK^T)V
$$
- $N$: 序列长度 (Sequence Length), $d$: 维度 (Dimension), 瓶颈： GPU 的计算单元 (Compute) 速度极快，但 HBM 的读写带宽 (Bandwidth) 有限。
- Standard Attention (Memory Bound)
    - 中间矩阵 $S$ ($QK^T$) 和 $P$ (Softmax结果) 的尺寸是 $N \times N$。它们太大了，SRAM 放不下，必须写回 HBM，然后再读回来。
    - 硬件执行流：
        - MatMul 1 (Q, K): 从 HBM 读取 $Q, K$ 到 SRAM。计算 $S = QK^T$。写回 HBM： 将巨大的 $N \times N$ 矩阵 $S$ 写入显存。 (高昂的 IO 开销)
        - Softmax (S): 从 HBM 读取整个 $S$ 到 SRAM。计算 $P = \text{softmax}(S)$ (需计算行最大值和行和)。写回 HBM： 将巨大的 $N \times N$ 矩阵 $P$ 写入显存。 (高昂的 IO 开销)
        - MatMul 2 (P, V): 从 HBM 读取 $P, V$ 到 SRAM。计算 $O = PV$。写回 HBM： 将结果 $O$ ($N \times d$) 写入显存。
- Flash Attention (IO Aware / Fused)：利用 Tiling (分块) 和 Online Softmax，将 $Q, K, V$ 切分成能在 SRAM 里放下的小块。在 SRAM 内部“熔合”所有计算，坚决不把 $N \times N$ 的中间矩阵写回 HBM。
    - 加载数据： 从 HBM 读取 $Q_i$ (当前块) 到 SRAM。初始化输出累加器 $O_i$、最大值统计量 $m_i$、分母和 $l_i$。
    - 循环加载 KV (Tiling): 遍历 $K, V$ 的所有分块 $K_j, V_j$：
        - 从 HBM 读取 $K_j, V_j$ 到 SRAM。
        - 在 SRAM 内计算 Attention Score: $S_{ij} = Q_i K_j^T$。
            - Online Softmax 更新:计算当前块的局部最大值。
            - Rescale (重缩放): 利用 Online Softmax 技巧，更新之前的 $O_i$（修正之前累加结果的缩放比例），并累加当前块的贡献。
        - 丢弃中间结果： $S_{ij}$ 用完即弃，不写回 HBM。
    - 写回结果： 循环结束后，$O_i$ 已经是最终正确的 Attention Output。写回 HBM： 仅将最终的 $O_i$ ($N \times d$) 写入显存。

In [1]:
import torch
import math

def standard_attention(Q, K, V):
    """
    标准 Attention 实现
    显存瓶颈：需要存储 N x N 的 scores 和 probs 矩阵
    """
    # 1. MatMul QK^T
    scores = torch.matmul(Q, K.transpose(-2, -1))
    
    # 2. Softmax (包含减去 max 操作以防溢出，虽然 PyTorch 内部做了)
    # scale factor
    d_k = Q.size(-1)
    scores = scores / math.sqrt(d_k)
    
    probs = torch.softmax(scores, dim=-1)
    
    # 3. MatMul PV
    output = torch.matmul(probs, V)
    return output

def minimal_flash_attention_simulation(Q, K, V, block_size_M=64, block_size_N=64):
    """
    Flash Attention 逻辑模拟 (Tiled + Online Softmax)
    核心：不生成完整的 N x N 矩阵，而是分块计算并动态 Rescale
    """
    N, d = Q.shape
    O = torch.zeros_like(Q)
    l = torch.zeros(N, 1)  # 存储分母 (行和)
    m = torch.ones(N, 1) * -torch.inf # 存储当前行的最大值
    
    scale = 1.0 / math.sqrt(d)

    # 外层循环：遍历 Q 的分块 (行)
    # 对应论文中的 Load Q_i from HBM to SRAM
    for i in range(0, N, block_size_M):
        Q_i = Q[i : i + block_size_M]
        
        # 初始化当前 block 的输出和统计量
        O_i = torch.zeros_like(Q_i)
        m_i = torch.ones(block_size_M, 1) * -torch.inf
        l_i = torch.zeros(block_size_M, 1)
        
        # 内层循环：遍历 K, V 的分块 (列)
        # 对应 Load K_j, V_j from HBM to SRAM
        for j in range(0, N, block_size_N):
            K_j = K[j : j + block_size_N]
            V_j = V[j : j + block_size_N]
            
            # 1. 计算当前块的 Attention Score (S_ij)
            S_ij = torch.matmul(Q_i, K_j.transpose(-2, -1)) * scale
            
            # 2. Online Softmax 核心逻辑
            # a. 找到当前块的局部最大值
            m_ij_block_max = torch.max(S_ij, dim=-1, keepdim=True)[0]
            
            # b. 更新全局最大值 m_new
            m_i_new = torch.maximum(m_i, m_ij_block_max)
            
            # c. 计算缩放系数 (Rescaling Factors)
            # P_ij = exp(S_ij - m_new) -> 当前块的分子
            P_ij_unscaled = torch.exp(S_ij - m_i_new)
            
            # l_new = l_old * exp(m_old - m_new) + current_block_sum
            # 这一步最关键：修正之前累加的 l (因为最大值变了)
            l_i_new = (l_i * torch.exp(m_i - m_i_new)) + torch.sum(P_ij_unscaled, dim=-1, keepdim=True)
            
            # d. 更新 Output
            # O_new = O_old * exp(m_old - m_new) + P_ij * V_j
            # 同样：修正之前累加的 O (因为最大值变了，分母暂不除，最后统一除)
            O_i = (O_i * torch.exp(m_i - m_i_new)) + torch.matmul(P_ij_unscaled, V_j)
            
            # e. 更新状态
            m_i = m_i_new
            l_i = l_i_new
        
        # 3. 最终归一化 (除以最终的分母和)
        # 这一步在写入 HBM 前做
        O[i : i + block_size_M] = O_i / l_i

    return O

# --- 验证等价性 ---
torch.manual_seed(42)
N, d = 128, 64  # 设置小一点方便观察，实际 N 会很大
Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

out_std = standard_attention(Q, K, V)
out_flash = minimal_flash_attention_simulation(Q, K, V, block_size_M=32, block_size_N=32)

# 检查误差
diff = (out_std - out_flash).abs().max()
print(f"Max Difference: {diff.item()}")

assert torch.allclose(out_std, out_flash, atol=1e-5), "Flash Attention 结果不一致！"
print("✅ Flash Attention 与 Standard Attention 数值等价！")

/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Max Difference: 2.980232238769531e-07
✅ Flash Attention 与 Standard Attention 数值等价！


符号定义：

设输入矩阵为 $Q, K, V \in \mathbb{R}^{N \times d}$。我们将矩阵划分为块（Blocks），以便加载到 SRAM 中：
- 行块 (Row Blocks) of $Q$: $Q_1, \dots, Q_{T_r}$，每个块大小为 $B_r \times d$。
- 列块 (Column Blocks) of $K, V$: $K_1, \dots, K_{T_c}$ 和 $V_1, \dots, V_{T_c}$，每个块大小为 $B_c \times d$。

我们在 SRAM 中维护以下 状态变量（针对当前的查询块 $Q_i$）：
- $O_i \in \mathbb{R}^{B_r \times d}$: 当前累积的输出结果（未归一化）。
- $m_i \in \mathbb{R}^{B_r}$: 当前行每一个位置的 局部最大值（用于数值稳定性）。
- $\ell_i \in \mathbb{R}^{B_r}$: 当前行每一个位置的 指数和（分母，Running Sum）。

算法过程 (Algorithmic Process)

算法分为外层循环（遍历 $Q$ 的行块）和内层循环（遍历 $K, V$ 的列块）。

初始化 (Initialization)对于每一个 $Q_i$（外层循环），在 SRAM 中初始化状态：$$\begin{aligned}
m_i^{(0)} &= [-\infty, \dots, -\infty]^T \\
\ell_i^{(0)} &= [0, \dots, 0]^T \\
O_i^{(0)} &= [0, \dots, 0]
\end{aligned}$$

第 $j$ 步迭代 (Step $j$ of Inner Loop)

当处理第 $j$ 个键值块 $(K_j, V_j)$ 时，我们需要将旧状态 $(m_i^{(j-1)}, \ell_i^{(j-1)}, O_i^{(j-1)})$ 更新为新状态 $(m_i^{(j)}, \ell_i^{(j)}, O_i^{(j)})$。

- 计算分块 Attention Score (On SRAM):加载 $Q_i, K_j$ 到 SRAM，计算当前块的相似度：$$S_{ij} = \frac{Q_i K_j^T}{\sqrt{d}} \in \mathbb{R}^{B_r \times B_c}$$
- 计算当前块的局部最大值:$$\tilde{m}_{ij} = \text{rowmax}(S_{ij}) \in \mathbb{R}^{B_r}$$
- 更新全局最大值 (Global Max Update):新的最大值是旧最大值与当前块最大值的较大者：$$m_i^{(j)} = \max(m_i^{(j-1)}, \tilde{m}_{ij})$$
- 计算重缩放因子 (Rescaling Factor):这是 Online Softmax 的核心。由于最大值 $m$ 变大了，我们需要计算一个修正系数，把基于旧最大值的数值“缩小”，以对齐到新最大值：$$\alpha_i^{(j)} = \exp(m_i^{(j-1)} - m_i^{(j)}) \in \mathbb{R}^{B_r}$$
-  计算当前块的非归一化概率 (Unscaled Probabilities):$$P_{ij} = \exp(S_{ij} - m_i^{(j)}) \in \mathbb{R}^{B_r \times B_c}$$
-  更新分母 (Running Sum Update):旧的分母 $\ell_i^{(j-1)}$ 需要乘以缩放因子 $\alpha$，再加上当前块的指数和：$$\ell_i^{(j)} = \ell_i^{(j-1)} \odot \alpha_i^{(j)} + \text{rowsum}(P_{ij})$$
-  更新输出 (Output Update):旧的输出 $O_i^{(j-1)}$ 同样需要重缩放，然后加上当前块的加权值：$$O_i^{(j)} = \text{diag}(\alpha_i^{(j)}) \cdot O_i^{(j-1)} + P_{ij} V_j$$
-  最终归一化 (Final Normalization)当内层循环结束（$j=T_c$）时，SRAM 中的 $O_i^{(T_c)}$ 是未归一化的 Attention 结果。我们需要除以最终的分母 $\ell_i^{(T_c)}$：$$O_i = \text{diag}(\ell_i^{(T_c)})^{-1} \cdot O_i^{(T_c)}$$最后将 $O_i$ 从 SRAM 写回 HBM。

$$
\text{Attention}(Q, K, V) = \frac{\sum_{j} e^{S_{ij}} \cdot V_j}{\sum_{j} e^{S_{ij}}}
$$
- 分子分母分开算，即独立维护了分子和分母
- 分子 (Numerator): 即我们在代码和公式中的 $O_i$。它实际上存储的是加权和：$O_i^{\text{unnorm}} \approx \sum e^{S_{ij}} \cdot V_j$
- 分母 (Denominator): 即我们在代码和公式中的 $\ell_i$。它存储的是归一化常数（Partition Function）：$\ell_i \approx \sum e^{S_{ij}}$
- 在循环中，$O_i$ 和 $\ell_i$ 都会不断地被更新（乘以重缩放因子 $\alpha$ 并加上新块的贡献），但它们始终保持着“未除”的状态。

### sdpa

- 新版 Transformers (v4.36+) 默认启用了 SDPA (Scaled Dot Product Attention) 也就是 Flash Attention 的 PyTorch 原生实现。为了极致的性能和显存优化，SDPA 默认是不计算并返回 $N \times N$ 的注意力矩阵的（因为这个矩阵非常大且慢）。
- 即使你在 forward 中指定了 output_attentions=True，在某些特定的版本/硬件组合下，SDPA 路径可能仍然返回了占位符（即一堆 None）。
- 需要强制模型使用传统的（Eager）注意力实现，这样它才会老老实实地把 $N \times N$ 矩阵算出来

```python
model = GPT2LMHeadModel.from_pretrained(model_name, attn_implementation="eager")
```